# Phase 3: Core Research — Lindblad Neural Operator (LNO) Training
**Stanford Code In Place — Main Physics-Informed Pipeline**

### Overview & Theory:
This notebook handles the primary training loop for our proposed **Lindblad Neural Operator (LNO)**. Standard neural operators struggle with open, dissipative systems. LNO incorporates structural non-unitary dynamics directly into the operator core via a parameterized Lindblad-like dissipation matrix ($D(\rho)$) and Gamma coupling parameters.

**Loss Function Matrix:**
$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{recon}} + \lambda_1 \mathcal{L}_{\text{physics}} + \lambda_2 \mathcal{L}_{\text{spectral}}$$
- $\mathcal{L}_{\text{physics}}$ enforces the continuity equation and mass conservation.
- $\mathcal{L}_{\text{spectral}}$ aligns the high-frequency decay profile with physical Kolmogorov laws.

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd

# Hardware checks and training seed initialization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

print(f"[+] Phase 3 Core Engine Active. Model training targeting: {device}")

[+] Phase 3 Core Engine Active. Model training targeting: cpu


In [ ]:
!mkdir -p models/checkpoints

In [ ]:
%%writefile models/spectral_kernel.py
"""
Yuánlǐ AI Research Laboratory
LNO-IonTransport Production Pipeline: High-Fidelity Fourier Spectral Kernel

Implements the 1D Fourier neural operator kernel parameterized by a truncated
set of complex-valued modes to perform global spatial convolutions efficiently.
"""

import torch
import torch.nn as nn
import torch.fft

class SpectralConv1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, modes: int):
        """
        Mathematical formulation:
            R_R(u)(x) = F^-1 ( R_hat(k) * F(u)(k) )
        where R_hat is a parameterized complex weight tensor restricting high frequencies.
        """
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes

        # Xavier Glorot initialization scale adapted for complex tensor allocations
        scale = 1.0 / (in_channels * out_channels)

        # Native PyTorch complex parameter initialization for seamless backpropagation
        self.weights = nn.Parameter(
            torch.complex(
                torch.randn(in_channels, out_channels, modes) * scale,
                torch.randn(in_channels, out_channels, modes) * scale
            )
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Input tensor shape x: [Batch, Channels, Nx]
        batch_size = x.shape[0]
        nx = x.size(-1)

        # 1. Transform spatial domain signals to frequency space
        x_ft = torch.fft.rfft(x, dim=-1)

        # 2. Allocate complex allocation tensor for output modes matching RFFT tracking
        out_ft = torch.zeros(
            batch_size,
            self.out_channels,
            nx // 2 + 1,
            dtype=torch.cfloat,
            device=x.device
        )

        # 3. Apply parameter weights tensor matrix multiplication over target modes spectrum
        # Tensor index mapping: b=batch, i=in_channels, o=out_channels, x=truncated_modes
        out_ft[:, :, :self.modes] = torch.einsum(
            "bix,iox->box",
            x_ft[:, :, :self.modes],
            self.weights
        )

        # 4. Inverse Fourier transform to map operators back to continuous spatial coordinates
        return torch.fft.irfft(out_ft, n=nx, dim=-1)

Writing models/spectral_kernel.py


In [ ]:
%%writefile models/dissipative_layer.py
"""
Yuánlǐ AI Research Laboratory
LNO-IonTransport Production Pipeline: Dissipative Lindblad Evolution Layer

Enforces explicit open-system non-unitary operator transformations to restrict
latent state trajectories to authentic thermodynamic irreversible relaxation bounds.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

class DissipativeEvolutionLayer(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.channels = channels

        # Coherent Drift Parameter Kernel for spatial shift transformations
        self.coherent_kernel = nn.Parameter(torch.randn(channels, channels, 5) * 0.02)

        # Multi-Channel Non-Unitary Jump Parameter Operators (L Matrix)
        self.jump_transform = nn.Parameter(torch.randn(channels, channels) * 0.01)

        # Spatial Smoothing Operator Weight to absorb continuous high-frequency noise spikes
        self.spatial_diffusion_weight = nn.Parameter(torch.ones(1, channels, 1) * 0.05)

    def _apply_lindblad_dissipation(self, x: torch.Tensor, gamma_field: torch.Tensor) -> torch.Tensor:
        """
        Evaluates the open-system dissipation mapping equation:
            D(x) = L * x * L^T - 0.5 * {L^T * L, x}
        """
        L = self.jump_transform
        L_dag_L = torch.mm(L.t(), L) # Construct positive-semidefinite matrix operator

        # Permute coordinates to compute pointwise interactions across channel spatial vectors
        x_permuted = x.permute(0, 2, 1) # [Batch, Nx, Channels]

        # Compute Sandwiched operation component: L * x * L^T
        sandwiched = torch.matmul(x_permuted, L.t())
        sandwiched = torch.matmul(sandwiched, L)

        # Compute Anti-commutator operational components: 0.5 * (L^T*L*x + x*L^T*L)
        anti_comm_left = torch.matmul(x_permuted, L_dag_L)
        anti_comm_right = torch.matmul(x_permuted, L_dag_L.t())
        anti_commutator = 0.5 * (anti_comm_left + anti_comm_right)

        dissipation_tensor = sandwiched - anti_commutator
        dissipation_tensor = dissipation_tensor.permute(0, 2, 1) # Return shape [Batch, Channels, Nx]

        # Broadcast environmental gamma field coefficients safely matching target spatial coordinates
        if gamma_field.dim() == 2:
            gamma_field = gamma_field.unsqueeze(1) # [Batch, 1, Nx]
        elif gamma_field.dim() == 1:
            gamma_field = gamma_field.view(-1, 1, 1)

        return gamma_field * dissipation_tensor

    def forward(self, x: torch.Tensor, gamma_field: torch.Tensor) -> torch.Tensor:
        # Path A: Coherent drift translation via periodic convolution padding
        padded_x = F.pad(x, (2, 2), mode='circular')
        coherent_out = F.conv1d(padded_x, self.coherent_kernel)

        # Path B: Thermodynamic open system Lindblad non-unitary dissipation step
        dissipative_out = self._apply_lindblad_dissipation(x, gamma_field)

        # Path C: Approximated spatial second-order continuous Laplacian bounds
        left_shift = torch.roll(x, shifts=-1, dims=-1)
        right_shift = torch.roll(x, shifts=1, dims=-1)
        laplacian_fields = left_shift - 2.0 * x + right_shift
        spatial_stabilization = self.spatial_diffusion_weight * laplacian_fields

        # Unified physical time-stepping integration loop tracking
        return x + coherent_out + dissipative_out + spatial_stabilization

    def compute_layer_entropy_penalty(self, x: torch.Tensor) -> torch.Tensor:
        """
        Tracks physical continuous mathematical Shannon entropy minimization behavior:
            S = - \sum p_i * log(p_i)
        """
        eps = 1e-8
        prob_distribution = F.softmax(x, dim=-1)
        entropy = -torch.sum(prob_distribution * torch.log(prob_distribution + eps), dim=-1)
        return -entropy.mean()


Overwriting models/dissipative_layer.py


In [ ]:
%%writefile models/lno_model.py
"""
Yuánlǐ AI Research Laboratory
LNO-IonTransport Production Pipeline: Master Lindblad Neural Operator Framework

Combines spatial Fourier spectral transformations with non-unitary open-system
Lindblad dissipation layers to optimize stable, thermodynamically valid predictions.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Any, List
from models.base_operator import BaseOperator
from models.spectral_kernel import SpectralConv1D
from models.dissipative_layer import DissipativeEvolutionLayer

class LindbladNeuralOperator(BaseOperator):
    def __init__(self, modes: int = 16, width: int = 64, depth: int = 4, in_channels: int = 6):
        super().__init__()
        self.width = width
        self.depth = depth
        self.modes = modes

        # 1. Coordinate Parameter Lifter Projection Head
        self.input_projection = nn.Linear(in_channels, width)

        # 2. Interleaved Parallel Spectral Operator Layers Sequence Blocks
        self.spectral_layers = nn.ModuleList([
            SpectralConv1D(width, width, modes) for _ in range(depth)
        ])

        self.pointwise_layers = nn.ModuleList([
            nn.Conv1d(width, width, 1) for _ in range(depth)
        ])

        # 3. Explicit Physics-Informed Dissipative Open Matrix Constraints Layers
        self.dissipative_layers = nn.ModuleList([
            DissipativeEvolutionLayer(width) for _ in range(depth)
        ])

        # 4. Standard Forward Space Trajectory Predictor Projection Head
        self.output_projection = nn.Sequential(
            nn.Linear(width, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

        # 5. Non-Equilibrium Inverse Dynamical Imaging Diagnostic Functional Head
        # Directly reconstructs unobserved systemic breakdown / instability regions fields
        self.instability_head = nn.Sequential(
            nn.Conv1d(width, width, 1),
            nn.GELU(),
            nn.Conv1d(width, 1, 1)
        )

    def forward(self, state: torch.Tensor, phi: torch.Tensor, flux: torch.Tensor,
                noise: torch.Tensor, dissipation: torch.Tensor, gamma: torch.Tensor) -> Dict[str, Any]:

        # Safety alignment processing over background execution inputs broadcasting vectors
        if gamma.dim() == 1:
            gamma = gamma.unsqueeze(-1)

        x = torch.stack(
            [
                state,
                phi,
                flux,
                noise,
                dissipation,
                gamma.repeat(1, state.shape[-1])
            ],
            dim=-1
        ) # Dimensional alignment matching layout: [Batch, Nx, 6]

        # Project spatial points parameters representation vectors to deep latent space dimensions
        x = self.input_projection(x)
        x = x.permute(0, 2, 1) # Shape: [Batch, Width, Nx]

        spectral_energy_history = []

        # Deep interleaved operators transformation execution pipeline
        for k in range(self.depth):
            # Compute parallel spatial transformations updates channels
            spectral_update = self.spectral_layers[k](x)
            pointwise_update = self.pointwise_layers[k](x)

            # Combine conservative propagation field states mapping vectors
            x_linear = spectral_update + pointwise_update

            # Constrain linear projections updates within explicit non-unitary open relaxation limits
            x = self.dissipative_layers[k](x_linear, gamma)

            # Non-linear scaling mapping constraints sequence
            if k < self.depth - 1:
                x = F.gelu(x)

            # Diagnostics processing: Track energy cascades tracking parameters continuously
            layer_energy = self.compute_spectral_energy(x)
            spectral_energy_history.append(layer_energy)

        # Execute Inverse Dynamical Imaging tracking maps reconstruction step
        instability_map = self.instability_head(x) # Output shape layout parameters: [Batch, 1, Nx]

        # Map variables configuration parameters tracking profiles back onto output projections grid spaces
        x = x.permute(0, 2, 1)
        predicted_evolution = self.output_projection(x) # Shape outcome targets values: [Batch, Nx, 1]

        return {
            "next_state": predicted_evolution.squeeze(-1),
            "instability_map": instability_map.squeeze(1),
            "spectral_energy": spectral_energy_history
        }

    def compute_thermodynamic_entropy_loss(self, prediction: torch.Tensor) -> torch.Tensor:
        """
        Calculates global unified distribution Shannon entropy constraints penalization boundaries.
        """
        eps = 1e-8
        prob_distribution = F.softmax(prediction, dim=-1)
        entropy_profile = -torch.sum(prob_distribution * torch.log(prob_distribution + eps), dim=-1)
        return -entropy_profile.mean()

    def get_spectral_stability_regularizer(self) -> torch.Tensor:
        """
        Calculates aggregate weight parameters Frobenius norms over active spectral
        operators grids to explicitly minimize numeric exploding modes expansion.
        """
        total_regularization_penalty = 0.0
        for module in self.modules():
            if isinstance(module, SpectralConv1D):
                total_regularization_penalty += torch.sum(torch.abs(module.weights) ** 2)
        return total_regularization_penalty

    def get_dissipative_coefficient_norm(self) -> torch.Tensor:
        """
        Tracks aggregate internal jump components regularizer weights variables profile data layers.
        """
        total_decay_norm = 0.0
        for module in self.modules():
            if isinstance(module, DissipativeEvolutionLayer):
                total_decay_norm += torch.sum(torch.abs(module.jump_transform))
        return total_decay_norm

Writing models/lno_model.py


In [ ]:
import os
os.makedirs("physics", exist_ok=True)

In [ ]:
%%writefile physics/pnp_equations.py
"""
LNO-IonTransport Production Pipeline: Stochastic Poisson–Nernst–Planck System

Governs the non-equilibrium electrodiffusive transport of ions, ionic transport,
thermal coupling, stochastic fluctuations, and dissipative transport evolution.
"""

import numpy as np
from typing import Dict, Any

class StochasticPNPSystem:
    """
    Research-Grade Stochastic Poisson–Nernst–Planck System

    PDE System:
        \partial c / \partial t = -\nabla \cdot J + \eta(x,t)
        J = -D(\nabla c + \beta \cdot c \cdot \nabla \phi)
        -\epsilon \nabla^2 \phi = \rho
    """

    def __init__(
        self,
        nx: int = 128,
        dx: float = 0.01,
        dt: float = 0.001,
        D: float = 0.1,
        epsilon: float = 0.05,
        beta: float = 1.0
    ):
        self.nx = nx
        self.dx = dx
        self.dt = dt
        self.D = D
        self.epsilon = epsilon
        self.beta = beta

    def solve_poisson(
        self,
        charge_density: np.ndarray,
        left_bc: float = 0.5,
        right_bc: float = 0.0,
        iterations: int = 120
    ) -> np.ndarray:
        """
        Solves the electrostatic potential using Successive Over-Relaxation (SOR).
        """
        phi = np.zeros(self.nx)
        phi[0] = left_bc
        phi[-1] = right_bc

        omega = 1.6

        for _ in range(iterations):
            for i in range(1, self.nx - 1):
                laplace_update = (
                    phi[i + 1]
                    + phi[i - 1]
                    + self.dx**2
                    * charge_density[i]
                    / self.epsilon
                ) / 2.0

                phi[i] = (
                    (1 - omega) * phi[i]
                    + omega * laplace_update
                )

        return phi

    def compute_flux(
        self,
        concentration: np.ndarray,
        potential: np.ndarray
    ) -> np.ndarray:
        """
        Computes the ionic flux combining Fickian diffusion and electrostatic drift.
        """
        grad_c = np.gradient(concentration, self.dx)
        grad_phi = np.gradient(potential, self.dx)

        flux = -self.D * (
            grad_c
            + self.beta
            * concentration
            * grad_phi
        )

        return flux

    def evolve(
        self,
        concentration: np.ndarray,
        noise: np.ndarray,
        gamma: float = 0.05
    ) -> Dict[str, np.ndarray]:
        """
        Advances the stochastic electrodiffusive field by a single temporal integration step.
        """
        phi = self.solve_poisson(concentration)

        flux = self.compute_flux(concentration, phi)

        div_flux = np.gradient(flux, self.dx)

        dissipation = gamma * concentration

        next_state = (
            concentration
            - self.dt * div_flux
            - self.dt * dissipation
            + np.sqrt(self.dt) * noise
        )

        next_state = np.clip(next_state, 1e-8, None)

        return {
            "state_next": next_state,
            "potential": phi,
            "flux": flux,
            "dissipation": dissipation
        }

Writing physics/pnp_equations.py


In [ ]:
%%writefile physics/lindblad_operator.py
"""
LNO-IonTransport Production Pipeline: Lindblad Dissipative Generator

Formulates non-unitary master equation dynamics adapted for stochastic
electrodiffusive transport systems.
"""

import numpy as np

class LindbladDissipativeGenerator:
    """
    Lindblad-Inspired Dissipative Operator

    Mathematical Structure:
        d\rho/dt = -i[H, \rho] + \sum_k (L_k \rho L_k^\dagger - 1/2 \{L_k^\dagger L_k, \rho\})
    """

    def __init__(self, gamma: float = 0.05, thermal_scale: float = 0.01):
        self.gamma = gamma
        self.thermal_scale = thermal_scale

    def commutator(self, H: np.ndarray, rho: np.ndarray) -> np.ndarray:
        return H @ rho - rho @ H

    def anti_commutator(self, A: np.ndarray, B: np.ndarray) -> np.ndarray:
        return A @ B + B @ A

    def dissipative_term(self, L: np.ndarray, rho: np.ndarray) -> np.ndarray:
        jump = L @ rho @ L.T
        damping = 0.5 * self.anti_commutator(L.T @ L, rho)
        return jump - damping

    def transport_generator(
        self,
        rho: np.ndarray,
        H: np.ndarray,
        jump_operators: list
    ) -> np.ndarray:
        coherent = -1j * self.commutator(H, rho)
        dissipative = np.zeros_like(rho, dtype=np.complex128)

        for L in jump_operators:
            dissipative += self.dissipative_term(L, rho)

        evolution = coherent + self.gamma * dissipative
        return evolution.real

    def stochastic_transport_step(self, state: np.ndarray, noise_strength: float = 0.01) -> np.ndarray:
        n = state.shape[0]
        rho = np.diag(state)
        H = np.diag(np.linspace(-1, 1, n))

        jump_operators = []
        for i in range(n):
            L = np.zeros((n, n))
            L[i, i] = self.gamma + self.thermal_scale
            jump_operators.append(L)

        evolution = self.transport_generator(rho, H, jump_operators)
        noise = noise_strength * np.random.randn(n)

        next_state = np.diag(rho) + np.diag(evolution) + noise
        next_state = np.clip(next_state, 1e-8, None)

        return next_state

Writing physics/lindblad_operator.py


In [ ]:
import os
os.makedirs("training", exist_ok=True)
os.makedirs("results/checkpoints", exist_ok=True)

In [ ]:
import inspect
from models.lno_model import LindbladNeuralOperator

print("[+] Inspected LNO __init__ Source Code:\n")
print(inspect.getsource(LindbladNeuralOperator.__init__))

[+] Inspected LNO __init__ Source Code:

    def __init__(self, modes: int = 16, width: int = 64, depth: int = 4, in_channels: int = 6):
        super().__init__()
        self.width = width
        self.depth = depth
        self.modes = modes
        
        # 1. Coordinate Parameter Lifter Projection Head
        self.input_projection = nn.Linear(in_channels, width)
        
        # 2. Interleaved Parallel Spectral Operator Layers Sequence Blocks
        self.spectral_layers = nn.ModuleList([
            SpectralConv1D(width, width, modes) for _ in range(depth)
        ])
        
        self.pointwise_layers = nn.ModuleList([
            nn.Conv1d(width, width, 1) for _ in range(depth)
        ])
        
        # 3. Explicit Physics-Informed Dissipative Open Matrix Constraints Layers
        self.dissipative_layers = nn.ModuleList([
            DissipativeEvolutionLayer(width) for _ in range(depth)
        ])
        
        # 4. Standard Forward Space Trajectory Predictor P

In [ ]:
import inspect
from training.losses import TotalLNOLoss

# forward method ka exact source code aur inputs print karte hain
print("[+] Inspected TotalLNOLoss Forward Source Code:\n")
print(inspect.getsource(TotalLNOLoss.forward))

[+] Inspected TotalLNOLoss Forward Source Code:

    def forward(self, pred: torch.Tensor, target: torch.Tensor, state_t: torch.Tensor,
                flux: torch.Tensor, noise: torch.Tensor, gamma: torch.Tensor) -> tuple:

        loss_recon = self.recon(pred, target)
        loss_diss = self.diss(state_t, pred)
        loss_entropy = self.entropy(pred, target)
        loss_spectral = self.spectral(pred)
        loss_residual = self.residual(state_t, pred, flux, noise, gamma)

        total_loss = (loss_recon + loss_diss + loss_entropy + loss_spectral + loss_residual)

        logs = {
            "total": total_loss.item(),
            "recon": loss_recon.item(),
            "diss": loss_diss.item(),
            "entropy": loss_entropy.item(),
            "spectral": loss_spectral.item(),
            "residual": loss_residual.item()
        }
        return total_loss, logs



In [ ]:
%%writefile training/trainer_lno.py
"""
LNO-IonTransport Production Pipeline: Physics-Informed Operator Training Loop Engine
"""

import torch
from tqdm import tqdm
from collections import defaultdict

class LNOTrainer:
    def __init__(self, model, optimizer, scheduler, criterion, device):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.criterion = criterion
        self.device = device

    def train_epoch(self, dataloader):
        self.model.train()
        total_loss = 0.0
        epoch_logs = defaultdict(float)

        pbar = tqdm(dataloader, desc="[Training Batch Step]", leave=False)
        for batch in pbar:
            if isinstance(batch, (tuple, list)):
                inputs = batch[0].to(self.device)
                targets = batch[1].to(self.device)
            elif isinstance(batch, dict):
                inputs = batch["inputs"].to(self.device)
                targets = batch["state_t"].to(self.device)

            # Unpack full 6-channel transport field configuration
            state = inputs[:, 0, :]
            phi = inputs[:, 1, :]
            flux = inputs[:, 2, :]
            noise = inputs[:, 3, :]
            dissipation = inputs[:, 4, :]
            gamma = inputs[:, 5, 0] # Uniform scalar coupling field per batch element

            self.optimizer.zero_grad()

            # Forward pass through Lindblad Operator Network
            outputs = self.model(
                state=state,
                phi=phi,
                flux=flux,
                noise=noise,
                dissipation=dissipation,
                gamma=gamma
            )

            # Compute Multi-Component Open System Physics-Informed Loss Tuple
            loss_val, batch_logs = self.criterion(
                pred=outputs["next_state"],
                target=targets,
                state_t=state,
                flux=flux,
                noise=noise,
                gamma=gamma
            )

            loss_val.backward()
            self.optimizer.step()

            # Metric accumulation loop
            total_loss += loss_val.item()
            for k, v in batch_logs.items():
                epoch_logs[k] += v

            pbar.set_postfix({
                "total": f"{loss_val.item():.4f}",
                "residual": f"{batch_logs.get('residual', 0):.4f}"
            })

        avg_loss = total_loss / len(dataloader)

        # Normalize aggregated physics logs across all training updates
        final_logs = {"running_loss": avg_loss}
        for k, v in epoch_logs.items():
            final_logs[f"avg_{k}"] = v / len(dataloader)

        return avg_loss, final_logs

    def validate(self, dataloader):
        self.model.eval()
        total_loss = 0.0

        with torch.no_grad():
            for batch in dataloader:
                if isinstance(batch, (tuple, list)):
                    inputs = batch[0].to(self.device)
                    targets = batch[1].to(self.device)
                elif isinstance(batch, dict):
                    inputs = batch["inputs"].to(self.device)
                    targets = batch["state_t"].to(self.device)

                state = inputs[:, 0, :]
                phi = inputs[:, 1, :]
                flux = inputs[:, 2, :]
                noise = inputs[:, 3, :]
                dissipation = inputs[:, 4, :]
                gamma = inputs[:, 5, 0]

                outputs = self.model(
                    state=state,
                    phi=phi,
                    flux=flux,
                    noise=noise,
                    dissipation=dissipation,
                    gamma=gamma
                )

                loss_val, _ = self.criterion(
                    pred=outputs["next_state"],
                    target=targets,
                    state_t=state,
                    flux=flux,
                    noise=noise,
                    gamma=gamma
                )
                total_loss += loss_val.item()

        return total_loss / len(dataloader)

Overwriting training/trainer_lno.py


In [ ]:
%%writefile training/optimizer.py
"""
LNO-IonTransport Production Pipeline: Optimizer Builder
"""

import torch

def build_optimizer(model: torch.nn.Module, config: dict) -> torch.optim.Optimizer:
    """
    Parses structural hyperparameters from configuration matrix
    and returns optimized execution engines.
    """
    train_cfg = config.get("training", {})
    optimizer_name = train_cfg.get("optimizer", "adamw").lower()
    lr = train_cfg.get("learning_rate", 1e-3)
    weight_decay = train_cfg.get("weight_decay", 1e-4)

    if optimizer_name == "adam":
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        raise ValueError(f"[-] Unsupported optimizer selection configuration: {optimizer_name}")

# Utils 

In [23]:
%%writefile utils/config_loader.py
"""
LNO-IonTransport Pipeline: Configuration Loader Utility
"""
import yaml
import os

def load_config(config_path="config.yaml"):
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"[-] Configuration file not found at {config_path}")

    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    print(f"[+] Configuration successfully loaded from {config_path}")
    return config

Overwriting utils/config_loader.py


In [26]:
%%writefile utils/checkpoint.py
"""
LNO-IonTransport Pipeline: Model Checkpointing System
"""
import torch
import os

def save_checkpoint(state, checkpoint_dir="results/checkpoints", filename="best_lno_model.pt"):
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir, exist_ok=True)

    filepath = os.path.join(checkpoint_dir, filename)
    torch.save(state, filepath)
    print(f"[+] Production-grade checkpoint securely saved to {filepath}")

def load_checkpoint(filepath, model, optimizer=None, scheduler=None, device="cuda"):
    if not os.path.exists(filepath):
        print(f"[-] No checkpoint found at {filepath}. Starting from scratch.")
        return None

    checkpoint = torch.load(filepath, map_location=device)
    model.load_state_dict(checkpoint['state_dict'])

    if optimizer and 'optimizer' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer'])
    if scheduler and 'scheduler' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler'])

    print(f"[+] Checkpoint loaded successfully from {filepath} (Epoch {checkpoint.get('epoch', 'unknown')})")
    return checkpoint

Overwriting utils/checkpoint.py
